# 01 - Data Preparation
In this notebook we fetch data from the Materials Project, or generate dummy data if an API key is not present, and save them as `extxyz` files.


In [1]:
import os
import numpy as np
from ase import Atoms
from ase.io import write
from ase.calculators.singlepoint import SinglePointCalculator
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()
os.makedirs('../data', exist_ok=True)


In [2]:
def generate_dummy_data(n_samples):
    """
    Fallback: Generates dummy structures when MP API key is missing.
    Uses SinglePointCalculator to attach energy and forces so that
    ASE's extxyz writer persists them correctly through the write/read cycle.
    """
    data = []
    for _ in range(n_samples):
        n_atoms = np.random.randint(3, 11)
        z = np.random.choice([1, 6, 8, 14, 26], size=n_atoms)
        pos = np.random.rand(n_atoms, 3) * 5.0
        cell = np.eye(3) * 10.0
        atoms = Atoms(numbers=z, positions=pos, cell=cell, pbc=True)
        energy = -float(n_atoms) * 2.5 + np.random.randn() * 0.1
        forces = np.random.randn(n_atoms, 3) * 0.5
        # Attach via SinglePointCalculator — this is the ONLY way ASE's
        # extxyz writer will persist energy and forces through write/read.
        # atoms.info['energy'] and atoms.arrays['forces'] are silently dropped.
        calc = SinglePointCalculator(atoms, energy=energy, forces=forces)
        atoms.calc = calc
        data.append(atoms)
    return data


MP_API_KEY = os.environ.get('MP_API_KEY')

if MP_API_KEY:
    print('MP_API_KEY found! Fetching data from Materials Project...')
    from mp_api.client import MPRester
    from pymatgen.io.ase import AseAtomsAdaptor

    dataset = []
    with MPRester(MP_API_KEY) as mpr:
        # num_chunks * chunk_size controls how many docs are fetched.
        # 'limit' was removed in newer mp-api versions.
        docs = mpr.materials.summary.search(
            num_elements=(1, 3),
            fields=['structure', 'energy_per_atom'],
            num_chunks=2,
            chunk_size=700,
        )

    for doc in tqdm(docs, desc='Processing structures'):
        if (
            getattr(doc, 'structure', None) is not None
            and getattr(doc, 'energy_per_atom', None) is not None
        ):
            atoms = AseAtomsAdaptor.get_atoms(doc.structure)
            energy = doc.energy_per_atom * len(atoms)
            forces = np.zeros((len(atoms), 3))
            calc = SinglePointCalculator(atoms, energy=energy, forces=forces)
            atoms.calc = calc
            dataset.append(atoms)

    print(f'Fetched {len(dataset)} structures from MP.')
    if len(dataset) < 1400:
        print(f'Padding with dummy data to reach 1400...')
        dataset.extend(generate_dummy_data(1400 - len(dataset)))
    else:
        dataset = dataset[:1400]
else:
    print('No MP_API_KEY found. Generating 1400 dummy structures...')
    dataset = generate_dummy_data(1400)


MP_API_KEY found! Fetching data from Materials Project...


Retrieving SummaryDoc documents:   0%|          | 0/1400 [00:00<?, ?it/s]

Processing structures:   0%|          | 0/1400 [00:00<?, ?it/s]

Fetched 1400 structures from MP.


> **Note on forces:** The Materials Project summary API does not return DFT forces.
> When using a real API key, forces are set to **zero** in the dataset.
> Force MAE benchmarking is only meaningful when using the dummy data path
> (which assigns random forces) or a separate dataset that includes actual DFT forces.


In [3]:
import random
random.seed(42)
random.shuffle(dataset)

train_data = dataset[:1000]
val_data   = dataset[1000:1200]
test_data  = dataset[1200:]

write('../data/train.extxyz', train_data)
write('../data/val.extxyz',   val_data)
write('../data/test.extxyz',  test_data)

n_tr = len(train_data)
n_v  = len(val_data)
n_te = len(test_data)
print('Train:', n_tr, '| Val:', n_v, '| Test:', n_te)


Train: 1000 | Val: 200 | Test: 200
